

## 1.1 · Load and denoise the data

In [2]:
from pathlib import Path
import html
import re
import unicodedata
import pandas as pd

paths = [folder / name for folder in (Path('data'), Path('../data'), Path.cwd(), Path('upload'))
         for name in ('research_abstracts.json', 'research_abstracts (1).json')]
path = next(p for p in paths if p.is_file())
raw = pd.read_json(path)
print('Input shape:', raw.shape)
print('Missing values:', raw.isna().sum().to_dict())
print('Original labels:', raw['Label'].value_counts().to_dict())

Input shape: (1138, 3)
Missing values: {'DOI': 4, 'Abstract': 0, 'Label': 0}
Original labels: {'classical': 900, 'ai': 238}


In [3]:
def clean_text(value):
    if pd.isna(value):
        return ''
    text = unicodedata.normalize('NFKC', html.unescape(str(value))).casefold()
    text = re.sub(r'</?[a-z][^>]*>', ' ', text)
    text = re.sub(r'(?:https?://|www\.)\S+|\b10\.\d{4,9}/\S+', ' ', text)
    text = re.sub(r'©[^\n]*$', ' ', text.strip())
    return ' '.join(re.findall(r'[^\W_]+', text))

papers = raw.copy()
papers['text'] = papers['Abstract'].map(clean_text)
papers['Label'] = papers['Label'].astype('string').str.strip().str.lower()
papers = papers.loc[papers['text'].ne('') & papers['Label'].notna() & papers['Label'].ne('')]
consistent = papers.groupby('text')['Label'].transform('nunique').eq(1)
print('Rows with conflicting labels:', (~consistent).sum())
papers = papers.loc[consistent].drop_duplicates('text').reset_index(drop=True)
print('Retained abstracts:', len(papers))
print('Retained labels:', papers['Label'].value_counts().to_dict())
print('Example:', papers.loc[0, 'text'][:180])

Rows with conflicting labels: 0
Retained abstracts: 1121
Retained labels: {'classical': 886, 'ai': 235}
Example: forward looking forecasting of the inflation rate could help the central bank and other government departments to better use monetary policy to stabilize prices and prevent the imp



## 1.2 · Select columns and model the features


In [4]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    papers['text'], papers['Label'].astype(str), test_size=0.20,
    stratify=papers['Label'], random_state=42)
model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
model.fit(X_train, y_train)
vectorizer, classifier = model.steps[0][1], model.steps[1][1]
terms = vectorizer.get_feature_names_out()
ranking = classifier.coef_[0].argsort()
print('Training matrix:', vectorizer.transform(X_train).shape)
for label, indices in zip(classifier.classes_, [ranking[:6], ranking[::-1][:6]]):
    print(label, 'associated features:', ', '.join(terms[indices]))

Training matrix: (896, 21571)
ai associated features: learning, neural, machine, machine learning, network, neural network
classical associated features: policy, monetary, the model, the predictions, effect, interest


## 1.3 · Evaluate learning and generalization


In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(model, X_train, y_train, cv=cv, scoring=['accuracy', 'f1_macro'])
dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
pred = model.predict(X_test)
for name, truth, guess in [
        ('Training', y_train, model.predict(X_train)),
        ('Test', y_test, pred), ('Majority rule on test', y_test, dummy.predict(X_test))]:
    print(f'{name:22s} n={len(truth):4d}  accuracy={accuracy_score(truth, guess):.3f}'
          f'  macro-F1={f1_score(truth, guess, average="macro"):.3f}')
for metric in ['accuracy', 'f1_macro']:
    values = scores['test_' + metric]
    print(f'CV {metric}: {values.mean():.3f} (fold SD {values.std():.3f})')
print(classification_report(y_test, pred, digits=3))
labels = ['ai', 'classical']
print(pd.DataFrame(confusion_matrix(y_test, pred, labels=labels),
                   index=['Actual ' + x for x in labels],
                   columns=['Predicted ' + x for x in labels]))

Training               n= 896  accuracy=0.993  macro-F1=0.990
Test                   n= 225  accuracy=0.907  macro-F1=0.862
Majority rule on test  n= 225  accuracy=0.791  macro-F1=0.442
CV accuracy: 0.886 (fold SD 0.014)
CV f1_macro: 0.820 (fold SD 0.025)
              precision    recall  f1-score   support

          ai      0.760     0.809     0.784        47
   classical      0.949     0.933     0.941       178

    accuracy                          0.907       225
   macro avg      0.854     0.871     0.862       225
weighted avg      0.909     0.907     0.908       225

                  Predicted ai  Predicted classical
Actual ai                   38                    9
Actual classical            12                  166


## 2.1 · Compare four additional classifiers


In [6]:
import numpy as np
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.svm import LinearSVC
from sklearn.linear_model import RidgeClassifier, SGDClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import make_scorer, recall_score

candidates = {
    'Q1 logistic regression': clone(model),
    'Linear SVM': make_pipeline(clone(vectorizer), LinearSVC(class_weight='balanced', random_state=42)),
    'Ridge classifier': make_pipeline(clone(vectorizer), RidgeClassifier(class_weight='balanced', solver='lsqr')),
    'Complement naive Bayes': make_pipeline(clone(vectorizer), ComplementNB()),
    'SGD modified Huber': make_pipeline(clone(vectorizer), SGDClassifier(loss='modified_huber', class_weight='balanced', random_state=42)),
}
metrics = {'accuracy': 'accuracy', 'macro_f1': 'f1_macro',
           'ai_recall': make_scorer(recall_score, pos_label='ai')}
def evaluate(estimator):
    result = cross_validate(estimator, X_train, y_train, cv=cv, scoring=metrics)
    return {key: result['test_' + key].mean() for key in metrics}

results = {name: evaluate(estimator) for name, estimator in candidates.items()}
print(pd.DataFrame(results).T.round(4).to_string())

                        accuracy  macro_f1  ai_recall
Q1 logistic regression    0.8862    0.8198     0.6701
Linear SVM                0.9007    0.8309     0.6225
Ridge classifier          0.8951    0.8263     0.6385
Complement naive Bayes    0.8025    0.4995     0.0586
SGD modified Huber        0.8638    0.8059     0.7552


## 2.2 · Enhance the features with supervised term weighting


In [7]:
class TermEvidence(BaseEstimator, TransformerMixin):
    def fit(self, X, y):
        present = X.sign()
        profiles = []
        for label in ['ai', 'classical']:
            counts = np.asarray(present[np.asarray(y) == label].sum(axis=0)).ravel() + 1
            profiles.append(counts / counts.sum())
        self.weights_ = np.abs(np.log(profiles[0] / profiles[1]))
        return self

    def transform(self, X):
        return X.multiply(self.weights_).tocsr()

enhanced = make_pipeline(clone(vectorizer), TermEvidence(),
                         clone(candidates['Linear SVM'].steps[-1][1]))
name = 'Evidence-weighted SVM'
candidates[name] = enhanced
results[name] = evaluate(enhanced)
table = pd.DataFrame(results).T.sort_values('macro_f1', ascending=False)
print(table.round(4).to_string())
print('Macro-F1 change vs ordinary SVM:', round(results[name]['macro_f1'] - results['Linear SVM']['macro_f1'], 4))
winner = table.index[0]
best = candidates[winner].fit(X_train, y_train)
predicted = best.predict(X_test)
print('Selected:', winner)
print(classification_report(y_test, predicted, digits=4))
print('Actual rows / predicted columns, ordered ai then classical:')
print(confusion_matrix(y_test, predicted, labels=['ai', 'classical']))

                        accuracy  macro_f1  ai_recall
Evidence-weighted SVM     0.9141    0.8470     0.6121
Linear SVM                0.9007    0.8309     0.6225
Ridge classifier          0.8951    0.8263     0.6385
Q1 logistic regression    0.8862    0.8198     0.6701
SGD modified Huber        0.8638    0.8059     0.7552
Complement naive Bayes    0.8025    0.4995     0.0586
Macro-F1 change vs ordinary SVM: 0.0161
Selected: Evidence-weighted SVM
              precision    recall  f1-score   support

          ai     0.9143    0.6809    0.7805        47
   classical     0.9211    0.9831    0.9511       178

    accuracy                         0.9200       225
   macro avg     0.9177    0.8320    0.8658       225
weighted avg     0.9196    0.9200    0.9155       225

Actual rows / predicted columns, ordered ai then classical:
[[ 32  15]
 [  3 175]]


In [8]:
import json
vec, weighting, svm = [step for _, step in best.steps]
effective = weighting.weights_ * svm.coef_.ravel()
vocabulary = {str(term): [float(idf), float(coef)] for term, idf, coef in
              zip(vec.get_feature_names_out(), vec.idf_, effective)}
casefold_overrides = {chr(i): chr(i).casefold() for i in range(0x110000)
                      if chr(i).casefold() != chr(i).lower()}
browser_model = dict(features=vocabulary, intercept=float(svm.intercept_[0]),
                     classes=svm.classes_.tolist(), fold=casefold_overrides)
payload = json.dumps(browser_model, separators=(',', ':'))
print('Exported', len(vocabulary), 'features for', winner)

Exported 21571 features for Evidence-weighted SVM


In [9]:
html_template = r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Researchly Research Abstract Classifier</title><style>
*{box-sizing:border-box}body{margin:0;color:#30323e;background:#fff;font:16px/1.55 "Source Sans 3",system-ui,sans-serif}main{max-width:770px;margin:clamp(14px,5vh,55px) auto;padding:0 clamp(14px,3vw,30px)}h1{font-size:clamp(33px,6vw,46px);line-height:1.13;letter-spacing:-1.5px;margin:0 0 22px}p{margin:0 0 20px}label{display:block;margin-bottom:6px;font-size:15px}textarea{display:block;width:100%;min-height:175px;padding:14px;border-radius:8px;border:1px solid transparent;background:#f0f2f6;font:inherit;font-size:15px;color:inherit;resize:vertical}button{margin-top:16px;padding:9px 15px;border:1px solid #c6c8cf;background:#fff;border-radius:8px;font:inherit;color:inherit;cursor:pointer}button:hover{background:#f3f4f6}button:focus-visible,textarea:focus-visible{outline:3px solid #397c80}h2{font-size:26px;margin:26px 0 12px}#answer{padding:16px;border-radius:8px;background:#e8f7ee;color:#196335}#message{color:#913e24;margin:8px 0}button:disabled{opacity:.6}
</style></head><body><main><h1>Researchly Research Abstract Classifier</h1><p>This app classifies research abstracts as either AI-related or classical research.</p><form><label for="abstract">Enter research abstract:</label><textarea id="abstract" required placeholder="Paste a research abstract here.">We train a neural network to identify patterns in clinical records. The machine learning model combines diagnostic histories with laboratory results to predict which patients are likely to require follow-up care.</textarea><button id="submit" disabled>Classify Abstract</button><div id="message" role="status">Loading classifier…</div></form><section id="result" hidden aria-live="polite"><h2>Prediction</h2><p id="answer"></p></section></main><script type="module">function predict(raw, model, decode) {
  const normalized = decode(raw).normalize("NFKC");
  let text = Array.from(normalized, ch => model.fold[ch] ?? ch.toLowerCase()).join("");
  text = text.replace(/<\/?[a-z][^>]*>/g, " ")
    .replace(/(?:https?:\/\/|www\.)\S+|\b10\.\d{4,9}\/\S+/g, " ").trim()
    .replace(/©[^\n]*$/u, " ");
  const tokens = (text.match(/[\p{L}\p{N}]+/gu) ?? []).filter(t => [...t].length >= 2);
  const counts = new Map();
  function count(term) {
    if (Object.hasOwn(model.features, term)) counts.set(term, (counts.get(term) ?? 0) + 1);
  }
  tokens.forEach((term, i) => {
    count(term);
    if (i) count(tokens[i - 1] + " " + term);
  });
  let squared = 0, dot = 0;
  for (const [term, n] of counts) {
    const [idf, coefficient] = model.features[term];
    const value = (1 + Math.log(n)) * idf;
    squared += value * value;
    dot += value * coefficient;
  }
  const score = model.intercept + (squared ? dot / Math.sqrt(squared) : 0);
  return {label: model.classes[Number(score > 0)], known: counts.size};
}
const form = document.querySelector("form"), field = document.querySelector("#abstract");
const result = document.querySelector("#result"), message = document.querySelector("#message");
const button = document.querySelector("#submit");
const decoder = document.createElement("textarea");
function decode(text) {
  return text.replace(/&(?:#(?:x[\da-f]+|\d+)|[a-z][a-z\d]*);?/gi, entity => {
    decoder.innerHTML = entity;
    return decoder.value;
  });
}
try {
  const model = MODEL_DATA;
  button.disabled = false;
  message.textContent = "";
  field.addEventListener("input", () => {result.hidden = true; message.textContent = "";});
  form.addEventListener("submit", event => {
    event.preventDefault();
    const {label, known} = predict(field.value, model, decode);
    if (!known) {
      result.hidden = true;
      message.textContent = "Please enter an abstract containing recognisable research terms.";
      return;
    }
    message.textContent = "";
    document.querySelector("#answer").textContent = "The abstract is classified as: " +
      (label === "ai" ? "artificial intelligence" : "classical research");
    result.hidden = false;
  });
} catch { message.textContent = "The classifier could not load. Refresh the page to try again."; }
</script></body></html>
'''
interface = Path('Researchly-Classifier.html')
interface.write_text(html_template.replace('MODEL_DATA', payload), encoding='utf-8')
print('Open', interface.name, 'in a web browser.')

Open Researchly-Classifier.html in a web browser.
